In [ ]:
# Cell 1 — Install dependencies
!pip install -q -U diffusers transformers accelerate
!pip install -q imageio imageio-ffmpeg

In [ ]:
# Cell 2 — HuggingFace login
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
login(token=UserSecretsClient().get_secret("HF_TOKEN"))
print("Logged in ✅")

In [ ]:
# Cell 3 — Upload image widget (run this, then upload your image)
import ipywidgets as widgets
from IPython.display import display
import io
from PIL import Image

upload = widgets.FileUpload(accept='image/*', multiple=False)
display(upload)

In [ ]:
# Cell 4 — Read uploaded image (run after uploading in Cell 3)
file_content = upload.value[0]['content']
image = Image.open(io.BytesIO(file_content)).convert('RGB')
image = image.resize((720, 480))  # CogVideoX-5b-I2V fixed resolution
print(f"Image loaded: {image.size}")
display(image)

In [ ]:
# Cell 5 — Load Stable Video Diffusion XT (fits T4 16GB in fp16, no quantization)
import torch
from diffusers import StableVideoDiffusionPipeline

pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16,
    variant="fp16",
)
pipe.enable_model_cpu_offload()
pipe.unet.enable_forward_chunking()
print("Model loaded ✅")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} / {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 6 — Generate video
import torch
import imageio

torch.cuda.empty_cache()

ANIMATE_PROMPT = "subtle atmospheric motion, gentle hair swaying, soft fabric rippling, fog drifting slowly, leaves rustling, no camera movement, no dramatic action"
NEGATIVE_PROMPT = "camera movement, zoom, pan, tilt, fast motion, shaking, text, watermark, blur"

output = pipe(
    image=image,
    prompt=ANIMATE_PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    num_frames=9,
    num_inference_steps=20,
    guidance_scale=6.0,
)

frames = output.frames[0]
imageio.mimsave("/kaggle/working/output.mp4", frames, fps=8)
print("Done! → /kaggle/working/output.mp4")

In [ ]:
# Cell 7 — Preview output
from IPython.display import Video
Video("/kaggle/working/output.mp4", embed=True)